# ✂️ 10 · Classical and Geometrical Segmentation
### Thresholds, watersheds, snakes, level sets and graph cuts: deciding which pixel belongs to what

**Part F · Segmenting, detecting, synthesizing** · Module 10 of 29 (core) · builds on 02, 03

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/10_segmentation_classical.ipynb)

| | |
|---|---|
| **Hardware target** | CPU |
| **Core frameworks** | NumPy, SciPy, scikit-image |
| **Prerequisites** | Module 02 (image formation), Module 03 (filtering) |
| **Estimated time** | 100–120 minutes |
| **02-740 topics covered** | global and Gaussian-mixture thresholding, region growing, mathematical morphology, watershed segmentation, SLIC superpixels, active contours (snakes), Chan–Vese level sets, graph cuts and normalized cut |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every image segmented below is produced by `course_toolkit.simulate_nuclei_image`, with the true per-pixel instance mask recorded by the simulator — so every method here can be checked against an exact answer key.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Threshold an image globally and by fitting a 1-D Gaussian mixture to its intensity histogram via
  expectation-maximization, and explain when each is appropriate.
- Explain region growing and quadtree-based split/merge segmentation.
- Apply the four basic morphological operations (dilation, erosion, opening, closing) and explain what
  each does to a binary mask's boundary.
- Run marker-controlled watershed segmentation, explain why markers reduce over-segmentation, and compute
  SLIC superpixels as a pre-segmentation step.
- Explain the active-contour ("snake") energy functional and watch a Chan–Vese level set evolve to find an
  object's boundary without any edge detection at all.
- Explain normalized cut as a spectral graph-partitioning problem and solve a small toy example by hand.

## 🗺️ Contents

1. What is segmentation, and why do so many methods exist?
2. Thresholding: global cutoffs and Gaussian-mixture EM
3. Region-based segmentation: growing, splitting and merging
4. Mathematical morphology and watershed segmentation
5. SLIC superpixels: useful over-segmentation
6. Active contours: the snake model
7. Chan–Vese level sets
8. Graph cuts and normalized cut

## 🧫 The biological question

Before anything in Modules 07–09 can run — before a single feature is computed or a single shape is
classified — something has to answer a much more basic question: *which pixels belong to this cell, and
which belong to its neighbor, or to the background?* That is segmentation, and bioimages make it hard in
specific, recurring ways: cells touch and overlap, illumination is uneven, noise competes with real
structure, and "the right answer" is a judgment call even for a trained human annotator. This notebook
walks through the classical (pre-deep-learning) toolbox for this problem, each method solving a different
piece of the puzzle.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 100
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def em_gaussian_mixture_1d(values: np.ndarray, k: int = 2, n_iter: int = 50, seed: int = 0):
    """From-scratch EM for a 1-D Gaussian mixture (the classic histogram-based way to pick a
    threshold: fit k components to the pixel-intensity histogram, e.g. background vs.
    foreground). Returns ``means``, ``stds``, ``weights`` and the final ``responsibilities``
    (n_values x k)."""
    rng = np.random.default_rng(seed)
    x = values.reshape(-1)
    means = rng.choice(x, size=k, replace=False).astype(float)
    stds = np.full(k, x.std() + 1e-6)
    weights = np.full(k, 1.0 / k)
    for _ in range(n_iter):
        # E-step: responsibilities via each component's Gaussian density
        dens = np.stack([
            weights[j] / (stds[j] * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((x - means[j]) / stds[j]) ** 2)
            for j in range(k)], axis=1)
        resp = dens / (dens.sum(axis=1, keepdims=True) + 1e-12)
        # M-step: weighted means/variances/weights
        nk = resp.sum(axis=0) + 1e-12
        means = (resp * x[:, None]).sum(axis=0) / nk
        stds = np.sqrt((resp * (x[:, None] - means) ** 2).sum(axis=0) / nk) + 1e-6
        weights = nk / len(x)
    return dict(means=means, stds=stds, weights=weights, responsibilities=resp)


def watershed_labels(image: np.ndarray, min_distance: int = 7) -> np.ndarray:
    """Marker-controlled watershed segmentation of a blobby intensity image: finds markers at
    local intensity maxima (one per expected object) and floods from them. Returns an integer
    label image (0 = background)."""
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed as _watershed
    smooth = ndi.gaussian_filter(image, 1.0)
    coords = peak_local_max(smooth, min_distance=min_distance, labels=smooth > smooth.mean())
    markers = np.zeros(image.shape, dtype=int)
    for i, (y, x) in enumerate(coords, start=1):
        markers[y, x] = i
    markers = ndi.label(ndi.binary_dilation(markers > 0, iterations=2))[0] if coords.size else markers
    elevation = -smooth
    return _watershed(elevation, markers=markers, mask=smooth > smooth.mean() * 0.6)


def slic_superpixels(image: np.ndarray, n_segments: int = 100) -> np.ndarray:
    """Simple Linear Iterative Clustering super-pixels (over-segmentation into compact,
    similar-intensity patches) — thin wrapper so the notebooks can call one function name."""
    from skimage.segmentation import slic
    rgb = np.stack([image] * 3, axis=-1)
    return slic(rgb, n_segments=n_segments, channel_axis=-1, start_label=1)


def toy_normalized_cut(affinity: np.ndarray) -> np.ndarray:
    """Split a small weighted graph (``affinity``, n x n, symmetric, non-negative) into two
    groups by the normalized-cut spectral relaxation: the Fiedler vector (second-smallest
    eigenvector) of the normalized graph Laplacian. Returns a length-n array of 0/1 group labels."""
    d = affinity.sum(axis=1)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(np.clip(d, 1e-9, None)))
    laplacian = np.diag(d) - affinity
    normalized = d_inv_sqrt @ laplacian @ d_inv_sqrt
    eigvals, eigvecs = np.linalg.eigh(normalized)
    fiedler = eigvecs[:, 1]  # eigvals[0] ~ 0 is the trivial constant eigenvector
    return (fiedler > np.median(fiedler)).astype(int)


def chan_vese_step(image: np.ndarray, phi: np.ndarray, mu: float = 0.2, dt: float = 0.5):
    """One explicit-Euler update of the Chan–Vese active-contour (level-set) evolution: moves
    the zero level set of ``phi`` to better separate the two regions of (near-)constant
    intensity it currently implies. Returns the updated ``phi``."""
    inside = phi > 0
    c1 = image[inside].mean() if inside.any() else image.mean()
    c2 = image[~inside].mean() if (~inside).any() else image.mean()
    grad_y, grad_x = np.gradient(phi)
    grad_mag = np.hypot(grad_y, grad_x) + 1e-6
    kappa = (np.gradient(grad_y / grad_mag, axis=0) + np.gradient(grad_x / grad_mag, axis=1))
    force = -(image - c1) ** 2 + (image - c2) ** 2
    return phi + dt * grad_mag * (mu * kappa + force)


def dice_coefficient(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Dice/F1 overlap of two binary masks: 2|A∩B| / (|A|+|B|)."""
    a, b = mask_a.astype(bool), mask_b.astype(bool)
    inter = (a & b).sum()
    denom = a.sum() + b.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1. What is segmentation, and why do so many methods exist?

**Image segmentation** is the process of assigning a label to every pixel such that pixels sharing a label
share some meaningful property — most simply, "object" vs. "background," or an integer id per individual
object (**instance segmentation**, which Module 11 revisits with deep learning). It is foundational: object
recognition, shape analysis, tracking and quantification all assume segmentation has already happened.

No single classical method dominates because bioimages fail in different ways: a clean, bimodal image
needs only a threshold; touching cells need watershed or active contours; a noisy image with intensity
inhomogeneity needs the statistical robustness of a Gaussian mixture or a globally optimal graph cut.
Real pipelines routinely combine several of these methods — exactly what you will see across this
notebook's sections.

## 2. Thresholding: global cutoffs and Gaussian-mixture EM

The simplest segmentation rule is a single intensity threshold $T$: $g(x,y)=1$ if $I(x,y)>T$, else $0$.
This works well when the image's intensity histogram is clearly **bimodal** — one mode for background, one
for foreground — and fails as soon as noise, uneven illumination, or overlapping intensity ranges blur that
separation.

A more principled way to *choose* $T$ fits the histogram with a **mixture of Gaussians**: assume every
pixel's intensity was drawn from one of $k$ Gaussian components (one per class), and use
**expectation-maximization (EM)** to fit the means, variances and mixture weights. EM alternates:

- **E-step**: given the current Gaussian parameters, compute each pixel's *responsibility* — the posterior
  probability it belongs to each component.
- **M-step**: given those responsibilities, re-estimate each component's mean, variance and weight as a
  responsibility-weighted average.

Once two neighboring Gaussians are fit, the threshold between them is simply the intensity where their two
densities cross — the point where it becomes more likely a pixel came from the *other* component.

In [ ]:
# ── 2.1 A deliberately uneven-illumination image, global threshold vs. EM threshold ──
sim = simulate_nuclei_image(n_cells=10, size=96, background=0.08, noise_sigma=0.035, seed=1)
image = sim["image"]
true_mask = sim["instance_mask"] > 0

global_thresh = image.mean() + 0.5 * image.std()
seg_global = image > global_thresh

em = em_gaussian_mixture_1d(image.ravel(), k=2, seed=SEED)
order = np.argsort(em["means"])   # component 0 = background (lower mean), component 1 = foreground
bg_idx, fg_idx = order
em_thresh = 0.5 * (em["means"][bg_idx] + em["means"][fg_idx])
seg_em = image > em_thresh

print(f"global threshold (mean + 0.5*std): {global_thresh:.3f}")
print(f"EM-fit component means: background={em['means'][bg_idx]:.3f}, foreground={em['means'][fg_idx]:.3f}")
print(f"midpoint EM threshold: {em_thresh:.3f}")
print(f"Dice vs. ground truth -- global: {dice_coefficient(seg_global, true_mask):.3f}, "
      f"EM: {dice_coefficient(seg_em, true_mask):.3f}")

In [ ]:
# ── 2.2 Visualize the histogram, the fitted mixture, and both segmentations ──
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
show_image(axes[0], image, "input image")
xs = np.linspace(image.min(), image.max(), 300)
hist_vals, bin_edges = np.histogram(image.ravel(), bins=60, density=True)
axes[1].bar((bin_edges[:-1] + bin_edges[1:]) / 2, hist_vals, width=np.diff(bin_edges), color="lightgray")
for j in (bg_idx, fg_idx):
    dens = em["weights"][j] / (em["stds"][j] * np.sqrt(2 * np.pi)) * \
        np.exp(-0.5 * ((xs - em["means"][j]) / em["stds"][j]) ** 2)
    axes[1].plot(xs, dens, label=f"component {j}")
axes[1].axvline(em_thresh, color="red", ls="--", label="EM threshold")
axes[1].legend(fontsize=7); axes[1].set_title("histogram + fitted mixture")
show_image(axes[2], seg_global, f"global threshold")
show_image(axes[3], seg_em, f"EM threshold")
plt.tight_layout(); plt.show()

🔍 **What just happened?** EM fit two Gaussians directly to the pixel-value
histogram with no hand-picked cutoff at all; the threshold fell out as the crossing point of the two fitted
densities. Because we have the simulator's ground-truth `instance_mask`, the Dice score tells us objectively
which threshold did better — not just which "looks" better.

✏️ **Your turn.** Repeat Section 2.1 with `noise_sigma=0.08` (much noisier).
Does the gap between the global-threshold Dice score and the EM-threshold Dice score grow or shrink? Why
would you expect EM's explicit noise model to matter more as noise increases?

### 📘 Lecture notes — determining the number of mixture components

`em_gaussian_mixture_1d` above was told `k=2` in advance. In general the right number of components is not
known ahead of time (is there a third, intermediate-intensity population — e.g. a dim halo around bright
nuclei?). The **Bayesian information criterion (BIC)** scores each candidate $k$ by trading off
goodness-of-fit (log-likelihood) against model complexity (number of parameters), penalizing an
unnecessarily large $k$ that would simply overfit noise; the $k$ with the lowest BIC is preferred.

## 3. Region-based segmentation: growing, splitting and merging

**Region growing** starts from one or more **seed** pixels (chosen either from strong evidence — a very
bright, clearly-foreground pixel — or by random sampling) and recursively absorbs neighboring pixels whose
intensity is similar enough to the growing region, stopping once no neighbor meets the similarity
criterion. **Splitting and merging** works top-down instead: start with the whole image as one region,
recursively **split** any region whose pixels are too inhomogeneous (commonly organized as a quadtree, each
node a square sub-region with a stored min/max intensity), then **merge** adjacent regions that turn out to
be similar enough after all. Both approaches use exactly two ingredients — a similarity rule and an
adjacency/connectivity rule — just applied in opposite directions.

In [ ]:
# ── 3.1 A simple region-growing segmentation from a single seed ──
def region_grow(image, seed_yx, tolerance):
    """Flood-fill from seed_yx, absorbing 4-connected neighbors within `tolerance` intensity"""
    seed_val = image[seed_yx]
    visited = np.zeros(image.shape, dtype=bool)
    stack = [seed_yx]
    visited[seed_yx] = True
    while stack:
        y, x = stack.pop()
        for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ny, nx = y + dy, x + dx
            if 0 <= ny < image.shape[0] and 0 <= nx < image.shape[1] and not visited[ny, nx]:
                if abs(image[ny, nx] - seed_val) < tolerance:
                    visited[ny, nx] = True
                    stack.append((ny, nx))
    return visited

single = simulate_nuclei_image(n_cells=1, size=64, shape_classes=("round",), seed=2)
seed_pixel = tuple(np.round(single["centroids"][0]).astype(int))
grown = region_grow(single["image"], seed_pixel, tolerance=0.18)
print(f"seed pixel value: {single['image'][seed_pixel]:.3f}  |  Dice vs. ground truth: "
      f"{dice_coefficient(grown, single['instance_mask'] > 0):.3f}")
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
show_image(axes[0], single["image"], "input (seed marked)")
axes[0].plot(seed_pixel[1], seed_pixel[0], "r+", markersize=12)
show_image(axes[1], grown, "region grown from seed")
plt.tight_layout(); plt.show()

✏️ **Your turn.** Try `tolerance=0.05` (too strict) and `tolerance=0.6` (too
loose). What failure mode does each produce — does the region stop growing too early, or does it "leak"
into the background? This tolerance plays the same role as the threshold $T$ in Section 2, just applied
relative to one seed pixel instead of globally.

## 4. Mathematical morphology and watershed segmentation

**Mathematical morphology** operates on a binary mask $A$ using a small "structuring element" $B$ (e.g. a
3×3 square): **dilation** $A \oplus B$ expands $A$ by adding every pixel whose structuring-element
neighborhood touches $A$; **erosion** $A \ominus B$ shrinks $A$, keeping only pixels whose whole
neighborhood lies inside $A$. Combining the two gives **opening** (erosion then dilation — smooths
contours, breaks thin connections, removes small protrusions) and **closing** (dilation then erosion —
fills small holes, connects narrow breaks).

**Watershed segmentation** treats an image's intensity surface as a topographic relief and "floods" it from
regional minima (or, for bright objects, from maxima of the *negated* image): wherever floods from two
different minima would meet, a **watershed line** is drawn — the segmentation boundary. Applied naively,
noise creates many spurious local minima and produces **over-segmentation**; the standard fix is
**marker-controlled watershed**: first smooth the image, then restrict flooding to start only from
confidently-identified markers (one per expected object), so stray noise-induced minima no longer create
their own regions.

In [ ]:
# ── 4.1 The four basic morphological operations ──
mask = single["instance_mask"] > 0
struct = np.ones((3, 3), dtype=bool)
dilated = ndi.binary_dilation(mask, structure=struct, iterations=2)
eroded = ndi.binary_erosion(mask, structure=struct, iterations=2)
opened = ndi.binary_opening(mask, structure=struct, iterations=2)
closed = ndi.binary_closing(mask, structure=struct, iterations=2)
show_image_grid([mask, dilated, eroded, opened, closed],
                 titles=["original mask", "dilation", "erosion", "opening", "closing"], ncols=5)
plt.show()

In [ ]:
# ── 4.2 Marker-controlled watershed on touching nuclei ──
touching = simulate_nuclei_image(n_cells=8, size=96, touching=True, seed=3)
labels_ws = watershed_labels(touching["image"], min_distance=6)
n_found = len(np.unique(labels_ws)) - 1   # subtract the background label
print(f"true number of cells: {len(touching['centroids'])}  |  watershed found: {n_found} regions")

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
show_image(axes[0], touching["image"], "touching nuclei (input)")
axes[1].imshow(labels_ws, cmap="nipy_spectral"); axes[1].set_title("marker-controlled watershed")
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** The simulator's `touching=True` flag deliberately
places cells closer together (`min_sep` shrinks) so some genuinely overlap and merge in the rendered image —
exactly the situation that breaks simple thresholding (one blob, not two) but that marker-controlled
watershed is specifically designed to pull apart, *provided* `min_distance` is tuned so each true cell gets
its own marker rather than noise creating extras or two close cells sharing one.

✏️ **Your turn.** Lower `min_distance` to `3` and re-run Section 4.2 — does the
found region count go up (closer to over-segmentation, extra spurious markers from noise) or down? This is
precisely the over-segmentation problem the lecture material warns about, made adjustable in one
parameter.

## 5. SLIC superpixels: useful over-segmentation

Sometimes *deliberate* over-segmentation is a feature, not a bug: grouping pixels into a modest number of
small, compact, locally-homogeneous **superpixels** can drastically reduce the number of units a later
algorithm (e.g. a graph cut, Section 8) has to reason about, while still respecting most true object
boundaries. **SLIC (simple linear iterative clustering)** does this with ordinary k-means, just using a
distance that combines color/intensity similarity with pixel-coordinate proximity, so that clusters stay
spatially compact instead of scattering across the whole image.

In [ ]:
# ── 5.1 SLIC superpixels at two different granularities ──
multi = simulate_nuclei_image(n_cells=6, size=96, seed=4)
sp_coarse = slic_superpixels(multi["image"], n_segments=40)
sp_fine = slic_superpixels(multi["image"], n_segments=150)
fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
show_image(axes[0], multi["image"], "input")
axes[1].imshow(sp_coarse, cmap="nipy_spectral"); axes[1].set_title(f"SLIC, ~40 segments")
axes[2].imshow(sp_fine, cmap="nipy_spectral"); axes[2].set_title(f"SLIC, ~150 segments")
for ax in axes[1:]: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

🔍 **What just happened?** Neither panel segments individual cells —
superpixels are a *pre-processing* step, not a final answer. Notice how coarse SLIC segments can straddle a
cell boundary (merging cell and background), while fine SLIC respects boundaries more faithfully but gives
you many more units to handle downstream. Choosing `n_segments` trades off exactly this.

## 6. Active contours: the snake model

A **snake** is a curve $v(s) = (x(s), y(s))$ for $s\in[0,1]$ that starts near an object's boundary and is
iteratively deformed to minimize an energy functional

$$E = E_{\text{internal}} + E_{\text{external}} + E_{\text{constraint}}$$

**Internal energy** $E_{\text{internal}} = \int \alpha(s)\lvert v'(s)\rvert^2 + \beta(s)\lvert
v''(s)\rvert^2\, ds$ keeps the curve smooth: the first term resists stretching (acts like an elastic
membrane), the second resists bending (acts like a stiff thin plate). **External energy** pulls the curve
toward image features — typically $E_{\text{edge}} = -\lvert\nabla I\rvert^2$, so the curve is attracted
to strong intensity gradients (object boundaries). The snake is initialized somewhere near the object and
iteratively updated (by gradient descent on $E$) until it settles into a local energy minimum, ideally right
on the true boundary.

In [ ]:
# ── 6.1 Run an active contour (snake) on a simulated nucleus ──
from skimage.segmentation import active_contour
from skimage.filters import gaussian as skimage_gaussian

snake_sim = simulate_nuclei_image(n_cells=1, size=96, shape_classes=("kidney",), seed=5)
smoothed = skimage_gaussian(snake_sim["image"], sigma=2)
cy, cx = snake_sim["centroids"][0]
theta = np.linspace(0, 2 * np.pi, 100)
init_radius = 42   # deliberately too large: the snake must shrink to find the true boundary
init_snake = np.stack([cy + init_radius * np.sin(theta), cx + init_radius * np.cos(theta)], axis=1)
final_snake = active_contour(smoothed, init_snake, alpha=0.015, beta=8, gamma=0.001, w_edge=1.5)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
show_image(ax, snake_sim["image"], "active contour (snake)")
ax.plot(init_snake[:, 1], init_snake[:, 0], "b--", label="initial contour")
ax.plot(final_snake[:, 1], final_snake[:, 0], "r-", lw=2, label="final contour")
ax.legend(fontsize=8); plt.show()

✏️ **Your turn.** Increase `beta` (the bending-resistance weight) to `30` and
re-run. The kidney shape has a concave "dent" — does a stiffer snake do a better or worse job of following
that concavity? This is the classic internal-vs-external energy trade-off: more rigidity resists noise but
also resists legitimately following a complex true boundary.

## 7. Chan–Vese level sets

Active contours as in Section 6 rely on image *gradients*, which struggle on low-contrast or noisy
boundaries. The **Chan–Vese model** instead asks a region-based question: if the image were split into
"inside" and "outside" a curve $C$, how different would the average intensities $c_1$ (inside) and $c_2$
(outside) have to be to explain the image well? Formally, Chan–Vese seeks the curve $C$ minimizing

$$F(c_1,c_2,C) = \mu\cdot\text{Length}(C) + \nu\cdot\text{Area}(\text{inside}(C))
    + \lambda_1\!\!\int_{\text{inside}(C)}\!\!|I-c_1|^2 + \lambda_2\!\!\int_{\text{outside}(C)}\!\!|I-c_2|^2$$

Rather than tracking the curve $C$ directly, Chan–Vese represents it implicitly as the **zero level set**
of a function $\phi(x,y)$ (positive inside, negative outside); evolving $\phi$ over time automatically
evolves $C = \{(x,y):\phi(x,y,t)=0\}$, including naturally splitting into multiple curves if the data
calls for it — something an explicit snake cannot do without extra bookkeeping.

In [ ]:
# ── 7.1 Evolve a Chan-Vese level set from a circle that does not match the object ──
cv_sim = simulate_nuclei_image(n_cells=1, size=96, shape_classes=("ring",), seed=6)
yy, xx = np.meshgrid(np.arange(96), np.arange(96), indexing="ij")
cy, cx = cv_sim["centroids"][0]
phi = 35 ** 2 - ((yy - cy) ** 2 + (xx - cx) ** 2)   # positive inside a circle of radius 35

snapshots = [phi.copy()]
for step in range(120):
    phi = chan_vese_step(cv_sim["image"], phi, mu=0.15, dt=0.4)
    if step in (9, 39, 119):
        snapshots.append(phi.copy())

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
titles = ["initial circle", "after 10 steps", "after 40 steps", "after 120 steps"]
for ax, snap, title in zip(axes, snapshots, titles):
    show_image(ax, cv_sim["image"], title)
    ax.contour(snap, levels=[0], colors="red", linewidths=2)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** The `"ring"` shape class is a genuinely hard case
for a single simple closed curve (a ring has a hole!) — watch whether the red contour manages to trace both
the inner and outer boundary of the ring, or only approximates the outer one. Chan–Vese's region-based
energy (not gradient-based) is exactly why it can handle low-contrast and even topologically tricky shapes
that a plain gradient-following snake would struggle with.

### 📘 Lecture notes — reading the Chan–Vese fitting term

For any candidate curve $C$, the "fitting value" $F_1(C)+F_2(C)$ (the two integral terms above, ignoring
length/area regularization) is close to zero *only* when $C$ sits right on the true boundary between two
roughly constant-intensity regions: if $C$ is entirely outside the object, $F_1>0$ (the "inside" region
wrongly includes background that does not match $c_1$); if entirely inside, $F_2>0$ instead; only when $C$
coincides with the real boundary are both terms simultaneously small. Minimizing $F_1+F_2$ is therefore
exactly equivalent to searching for that boundary.

## 8. Graph cuts and normalized cut

A different strategy represents the whole image as a weighted graph $G=(V,E)$: each vertex is a pixel (or,
more practically, a superpixel from Section 5), and each edge weight encodes how similar two vertices are.
Segmentation becomes **graph partitioning**: split $V$ into two disjoint sets $A,B$ minimizing the **cut**
$\text{cut}(A,B)=\sum_{u\in A,v\in B} w(u,v)$ — the total similarity severed by the split. Plain min-cut
has a pathological tendency to isolate a handful of outlier nodes into a tiny set $B$, trivially minimizing
the cut without finding a meaningful partition. The **normalized cut (Ncut)** fixes this by dividing by each
side's total connection to the whole graph:

$$\text{Ncut}(A,B) = \frac{\text{cut}(A,B)}{\text{assoc}(A,V)} + \frac{\text{cut}(A,B)}{\text{assoc}(B,V)}$$

Shi & Malik (2000) showed that minimizing Ncut can be relaxed to an eigenvector problem: compute the
**Fiedler vector** (the eigenvector of the normalized graph Laplacian with the second-smallest eigenvalue —
the smallest eigenvalue's eigenvector is always the trivial constant vector) and split vertices by its sign
(or median) — exactly what Module 09's spectral intuition (eigenvectors of a covariance/Laplacian matrix
capturing the data's dominant structure) predicted would be useful again here.

In [ ]:
# ── 8.1 A tiny toy affinity graph: two obvious groups plus one ambiguous node ──
# nodes 0,1,2 should form one group; nodes 3,4,5 another; node 6 sits "between" the two groups
affinity = np.array([
    [0, 9, 8, 1, 0, 0, 3],
    [9, 0, 7, 0, 1, 0, 2],
    [8, 7, 0, 0, 0, 1, 3],
    [1, 0, 0, 0, 9, 8, 3],
    [0, 1, 0, 9, 0, 7, 2],
    [0, 0, 1, 8, 7, 0, 3],
    [3, 2, 3, 3, 2, 3, 0],
], dtype=float)
groups = toy_normalized_cut(affinity)
print("normalized-cut group assignment for nodes 0-6:", groups)

In [ ]:
# ── 8.2 Visualize the toy graph, colored by the Ncut partition ──
import itertools
positions = {0: (0, 1), 1: (0.3, 1.6), 2: (-0.3, 1.6),
             3: (2, -1), 4: (2.3, -1.6), 5: (1.7, -1.6), 6: (1, 0)}
fig, ax = plt.subplots(figsize=(5, 4))
for i, j in itertools.combinations(range(7), 2):
    if affinity[i, j] > 0:
        xi, yi = positions[i]; xj, yj = positions[j]
        ax.plot([xi, xj], [yi, yj], color="gray", lw=affinity[i, j] / 3, zorder=1)
colors = ["tab:blue" if g == 0 else "tab:orange" for g in groups]
for i, (x, y) in positions.items():
    ax.scatter(x, y, s=400, color=colors[i], zorder=2, edgecolor="black")
    ax.annotate(str(i), (x, y), ha="center", va="center", zorder=3, fontweight="bold")
ax.set_title("Normalized-cut partition of a toy 7-node graph"); ax.axis("off")
plt.show()

🔍 **What just happened?** The affinity matrix was built by hand so nodes
0–2 and 3–5 are each strongly mutually connected while node 6 connects moderately to everyone — a graph
version of "two obvious cell clusters plus one ambiguous boundary pixel." Normalized cut's eigenvector
relaxation found the natural 0–2 / 3–5 split and assigned node 6 to whichever side it is, on balance, more
strongly associated with — exactly the behavior a plain min-cut (which would be tempted to isolate node 6
alone, since its edges are all individually weak) is prone to getting wrong.

✏️ **Your turn.** Change `affinity[6, :]` and `affinity[:, 6]` so that node 6
connects *only* to node 3 with a very strong weight (say, 15) and weakly or not at all to everyone else.
Re-run — does normalized cut now assign node 6 to the 3–5 group instead? This should feel exactly like
image segmentation: node 6 is "a pixel," and its group assignment depends entirely on which neighboring
pixels it is most similar to.

📌 **Key takeaways**

- Thresholding is simple and fast but assumes a (roughly) bimodal histogram; fitting a Gaussian mixture via EM finds a principled threshold even under noise and uneven illumination, at the cost of needing to choose (or select via BIC) the number of components.
- Region growing and split/merge both rely on just two ingredients — similarity and adjacency — applied bottom-up or top-down respectively.
- Dilation/erosion/opening/closing are the basic morphological vocabulary that watershed (and many cleanup steps elsewhere in this course) are built from; marker-controlled watershed is the standard fix for watershed's over-segmentation problem.
- SLIC superpixels trade segmentation granularity for a smaller, more tractable set of units for downstream algorithms.
- Active contours (gradient-driven) and Chan–Vese level sets (region-statistics-driven) both iteratively deform a curve to minimize an energy functional, but behave very differently on low-contrast, noisy, or topologically complex objects.
- Graph-cut segmentation reframes the problem as graph partitioning; normalized cut avoids plain min-cut's bias toward isolating small outlier sets by normalizing the cut cost by each side's total connectivity, solvable via a Laplacian eigenvector.

✅ **Checkpoint**

1. Why does marker-controlled watershed reduce over-segmentation compared to running watershed directly
   on a noisy image?
   <details><summary>Answer</summary>Unrestricted watershed floods from *every* local minimum, including
   spurious ones created by noise; restricting flooding to start only from a small set of
   confidently-identified markers (one per expected object, after smoothing) removes those spurious
   starting points.</details>
2. A plain min-cut partition of an image graph isolates a single noisy pixel as its own segment. Why does
   normalized cut avoid this?
   <details><summary>Answer</summary>Plain min-cut cost only counts the severed edge weights, which are
   trivially small for an isolated pixel's few weak edges; Ncut instead divides by that pixel's total
   association to the whole graph, which is also small, so the ratio no longer rewards isolating
   low-connectivity nodes.</details>
3. Chan–Vese's fitting energy is near zero only when the candidate curve matches the true boundary. Why
   does this NOT require computing an image gradient the way an active-contour snake does?
   <details><summary>Answer</summary>Chan–Vese's energy compares each region's *average intensity* to the
   actual pixel intensities within it (a region statistic), rather than looking for strong intensity
   *gradients* at the boundary — so it can succeed even on blurry, low-gradient boundaries where
   gradient-based methods have little signal to follow.</details>

🧠 **Questions & discussion**

- Section 2 showed EM-fit thresholding beating a fixed global threshold under heavier noise. Can you think of an imaging scenario (think about uneven illumination across a large field of view) where even EM thresholding, applied globally to the whole image, would still fail, and a *local* adaptive threshold would be needed instead?
- SLIC superpixels and marker-controlled watershed both reduce over-segmentation risk, but by different mechanisms. Could you use SLIC superpixels *as* the markers for a subsequent watershed pass? What would that buy you?
- Every method in this notebook needed at least one hand-tuned parameter (a tolerance, a `min_distance`, `alpha`/`beta`, `mu`). How would you go about choosing these systematically on simulated data where ground truth (Dice score) is available, versus on a brand-new real dataset where it is not?

🚀 **Bonus — going further**

- Combine Sections 4 and 5: compute SLIC superpixels, then build an affinity graph between *superpixels* (rather than the toy 7-node graph of Section 8) using mean intensity similarity, and apply `toy_normalized_cut` to it.
- Add an area regularization term to the Chan–Vese update in Section 7 (the $\nu$ term in the energy) and see how a strong area penalty changes the final contour.
- Quantitatively sweep the watershed `min_distance` parameter from 2 to 15 against the simulator's known cell count, and plot found-region-count vs. `min_distance` to find the value that best matches ground truth for a given amount of touching.

📝 **Lab note**

```text
Module 10 — Classical and geometrical segmentation
Date: ____________
Dice, global threshold vs. EM threshold (noisy image): ____ vs. ____
Watershed: true cell count ____  found regions ____  (min_distance = ____)
Did the Chan-Vese contour recover the ring's inner boundary?  yes / no
Normalized-cut assignment for the ambiguous node, before/after re-wiring: ____ / ____
```

📚 **References**

- Otsu, N. (1979). A threshold selection method from gray-level histograms. *IEEE Transactions on Systems, Man, and Cybernetics*, 9(1), 62–66.
- Dempster, A. P., Laird, N. M. & Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society B*, 39(1), 1–38.
- Vincent, L. & Soille, P. (1991). Watersheds in digital spaces: an efficient algorithm based on immersion simulations. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 13(6), 583–598.
- Achanta, R. et al. (2012). SLIC superpixels compared to state-of-the-art superpixel methods. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 34(11), 2274–2282.
- Kass, M., Witkin, A. & Terzopoulos, D. (1988). Snakes: Active contour models. *International Journal of Computer Vision*, 1(4), 321–331.
- Chan, T. F. & Vese, L. A. (2001). Active contours without edges. *IEEE Transactions on Image Processing*, 10(2), 266–277.
- Shi, J. & Malik, J. (2000). Normalized cuts and image segmentation. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 22(8), 888–905.

➡️ **Next.** Module 11 asks the same segmentation question again, but hands it to a neural network: fully convolutional networks, U-Net, and the weighted losses that let a CNN learn to separate touching cells — the deep-learning counterpart to this notebook's classical toolbox.